[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quadrics/spherical_quadrics/spherical_quadrics.ipynb)

# A Conic on the Sphere

This example draws an ellipse on a sphere and checks the two properties that make it one. On a sphere an ellipse is where a cone through the centre cuts the surface; like an ellipse in the plane it has two foci, and the sum of the distances, along great circles, from the foci to any point of it is the same. Seen through its tangent great circles instead of its points, it has a twin property: the product of the sines of the foci's distances to its tangents is the same for every tangent. The same form, evaluated over the whole sphere, has as its level sets the polhodes of a spinning body.

In the geometric algebra of the sphere, points are antivectors and planes through the centre, the great circles, are vectors. The conic is a polarity, a map from a point to its polar plane, `Plane <- Point`, a sum of the coordinate planes' dyads weighted by its eigenvalues, and the curve is where a point lies on its own polar. The polar plane of a point of the curve is its tangent great circle, and the tangents lie on the dual conic, the inverse map from planes to points. One rotor turns both.

In the notation of rigid-body dynamics the form's level sets read as Poinsot's polhodes, the curves traced on the momentum sphere where it meets the inertia ellipsoid.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext, stack
from numga.algebras import Spherical3D as ga
from examples.util.animation import save_figure
from examples.quadrics.spherical_quadrics import render
from examples.quadrics.spherical_quadrics.core import points

np.set_printoptions(precision=4, suppress=True)

mv = NumpyContext(ga).multivector
# Points on yz, zx, xy: the poles of the planes x, y, z.
Point = ga.gatype.antivector()                      # Antivector-
Plane = ga.gatype.plane()                           # Vector+

## 1. The conic and its dual

In its principal axes the polarity is diagonal, each coordinate plane weighted by an eigenvalue; with two positive and one negative, the cone is real and cuts the sphere in an oval. A rotor places it, acting on the polarity and on its dual, the inverse map, by the same sandwich.

In [ ]:
# The polarity's eigenvalues on the planes x, y and z, and the turn that places it, in its two planes.
eigenvalues = np.array([1.0, 0.4, -0.8])
turn_xy, turn_yz = 0.25, 0.15
n_curve = 200

In [ ]:
def oval(eigenvalues: np.ndarray, t: np.ndarray) -> tuple[Point, Point, float]:
    """The spherical conic of a diagonal polarity with eigenvalues l1 > l2 > 0 > l3 has these points, its two foci,
    and its semi-major arc."""
    l1, l2, l3 = eigenvalues
    # The cone meets the sphere above the ellipse x**2 / x0**2 + y**2 / y0**2 == 1 of the xy plane; its semi-axes
    # as arcs, the semi-minor along x and the semi-major along y, and the focal arc along the major axis y.
    x0 = np.sqrt(-l3 / (l1 - l3))
    y0 = np.sqrt(-l3 / (l2 - l3))
    theta_b = np.arcsin(x0)
    theta_a = np.arcsin(y0)
    theta_c = np.arccos(np.cos(theta_a) / np.cos(theta_b))
    x, y = x0 * np.cos(t), y0 * np.sin(t)
    curve = points(np.stack([x, y, np.sqrt(1.0 - x**2 - y**2)], axis=-1))      # [curve] Point-
    foci = points(np.array([[0.0, np.sin(theta_c), np.cos(theta_c)], [0.0, -np.sin(theta_c), np.cos(theta_c)]]))   # [2] Point-
    return curve, foci, theta_a


# In its principal axes the polarity is the sum of the coordinate planes' dyads, weighted by its eigenvalues.
planes = stack([mv.x, mv.y, mv.z])                                             # [3] Plane+
principal = (planes * (planes & Point) * eigenvalues).sum(axis=0)              # [] Plane+ <- Point-
rotor = (mv.xy * turn_xy + mv.yz * turn_yz).exp()                              # [] Rotor+
conic = rotor >> principal(rotor << Point)                                     # [] Plane+ <- Point-
dual = rotor >> principal.solve(rotor << Plane)                                # [] Point- <- Plane+
unturned, unturned_foci, semi_major = oval(eigenvalues, np.linspace(0, 2 * np.pi, n_curve))
curve, foci = rotor >> unturned, rotor >> unturned_foci                        # [curve] Point-, [2] Point-

## 2. Two foci, and their twins on the tangents

The distance between two unit points of the sphere is the arc whose cosine is their inner product, `point.dot(focus)`, and the sum of the arcs from the two foci to any point of the oval is twice its semi-major arc. The polar plane of each point of the oval is its tangent great circle; joined with a focus it gives the sine of the focus's distance to that circle, and the product for the two foci is the same for every tangent.

In [ ]:
first, second = foci
focal_sum = curve.dot(first).clip(-1.0, 1.0).arccos() + curve.dot(second).clip(-1.0, 1.0).arccos()   # [curve] Scalar+
tangents = conic(curve).normalized()                                           # [curve] Plane+
dual_product = (first & tangents).norm() * (second & tangents).norm()          # [curve] Scalar+

In [ ]:
print("sum of the focal distances:", focal_sum.to_array().mean(), "  twice the semi-major arc:", 2 * semi_major)
print("product of the sines to the tangents:", dual_product.to_array().mean())

## 3. Three views

The left panel shows the oval in gold, its antipodal twin dashed, the foci in red and the geodesics from them to one point, over the polarity's form shaded on the sphere. The middle panel shows the oval as the envelope of its tangent great circles, each with its pole, and the dual conic the poles trace, in magenta. The right panel shows the cone through the oval, and the level sets of the form on the sphere, the polhodes.

In [ ]:
# The point the geodesics from the foci run to.
sample_index = 25

In [ ]:
sample = curve[sample_index]
save_figure(render.draw_spherical_conic(curve, foci, sample, tangents, conic), "spherical_quadrics");

In [ ]:
# checks
# Every point of the oval lies on its own polar; the focal sum is twice the semi-major arc; every tangent
# lies on the dual conic; the product of the sines is constant.
np.testing.assert_allclose((curve & conic(curve)).to_array(), 0.0, atol=1e-12)
np.testing.assert_allclose(focal_sum.to_array(), 2.0 * semi_major, atol=1e-11)
np.testing.assert_allclose((tangents & dual(tangents)).to_array(), 0.0, atol=1e-12)
np.testing.assert_allclose(dual_product.to_array(), dual_product.mean().to_array(), atol=1e-14)